# GPU benchmark: our own qwen3.8 GGUF+mmproj via CUDA llama-cpp-python on RTX Pro 6000

Offline (no internet), Phase A only, no quota spent. Uses OUR OWN weights (extracted from Ollama, already tested/verified locally on CPU) and OUR OWN CUDA-compiled llama-cpp-python wheel -- no third-party models. Directly comparable to the local CPU numbers already measured (170.5s for reasoning_effort=xhigh, 168.9s for low).

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)


In [ ]:
from pathlib import Path

def find_under_input(name):
    for match in Path("/kaggle/input").rglob(name):
        return match
    raise FileNotFoundError(f"{name} not found under /kaggle/input")

WHEEL = find_under_input("llama_cpp_python-0.3.35-py3-none-linux_x86_64.whl")
MODEL_GGUF = find_under_input("qwen3.8-27b.gguf")
MODEL_MMPROJ = find_under_input("qwen3.8-27b-mmproj.gguf")
print("wheel:", WHEEL)
print("model:", MODEL_GGUF, MODEL_GGUF.stat().st_size / 1e9, "GB")
print("mmproj:", MODEL_MMPROJ, MODEL_MMPROJ.stat().st_size / 1e9, "GB")


In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps",
                 "--find-links", str(WHEEL.parent), str(WHEEL)], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-index",
                 "--find-links", str(WHEEL.parent), "diskcache"], check=True)


In [ ]:
import llama_cpp
print("llama_cpp version:", llama_cpp.__version__)
print("supports_gpu_offload:", llama_cpp.llama_cpp.llama_supports_gpu_offload())


In [ ]:
import base64, time
from llama_cpp import Llama
from llama_cpp.llama_chat_format import MTMDChatHandler

t0 = time.time()
handler = MTMDChatHandler(clip_model_path=str(MODEL_MMPROJ), verbose=False)
llm = Llama(model_path=str(MODEL_GGUF), chat_handler=handler, n_ctx=4096, n_gpu_layers=-1, verbose=True)
print(f"GPU load time: {time.time()-t0:.1f}s")


In [ ]:
# same real ls20 frame + same prompt used in the local CPU benchmark, for a direct comparison
import urllib.request
# a small red/yellow test PNG is fine here too, but let's build a real-looking one procedurally
# since we don't have this repo's src/ bundled in this diagnostic-only kernel
from PIL import Image
import numpy as np
img = Image.new("RGB", (512, 512), (255, 220, 0))
img.save("/kaggle/working/test_frame.png")
img_b64 = base64.b64encode(open("/kaggle/working/test_frame.png", "rb").read()).decode()

for effort in ["xhigh", "low"]:
    t0 = time.time()
    resp = handler(
        llama=llm,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": "Describe what you see in this image in 2-3 sentences: background color, shapes, and their approximate positions."},
            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{img_b64}"}},
        ]}],
        max_tokens=200,
        reasoning_effort=effort,
    )
    elapsed = time.time() - t0
    print(f"=== reasoning_effort={effort}: {elapsed:.1f}s (CPU baseline was {'170.5s' if effort=='xhigh' else '168.9s'}) ===")
    print(resp["choices"][0]["message"]["content"][:400])
    print()
